# Tutorial about fluopy - add fluorophore and transition data

Here we provide some hints on extending fluoropy with new fluorophore and transition data.

In [ ]:
from pathlib import Path

import numpy as np

import fluopy

## Naming conventions

|name|meaning|
|---|---|
|SingleState|photophysical state fo a single fluorophore|
|PairedState|Two SingleStates of paired fluorophores as in donor and acceptor pairs|
|Transition|constant and variable attributes of photophysical transition|
|combined states|combinations of SingleStates depending on number of fluorophores|
|combined state transition|transitions between combined states|
|realizable|theoretically possible|
|resample|change the frame integration time|

## Adding a fluorophore

A fluorophore that is not included with Fluopy can be defined in a notebook.

1. Create an S0 absorption Spectrum from arrays or a CSV file. Also create an
   emission Spectrum when bandpass filtering or energy-transfer calculations are
   required.
2. Create a FluorophoreData object containing the spectra and photophysical constants.
3. Pass the FluorophoreData object to Fluorophore
4. Use FluorophoreSystem.load_transitions() to derive transitions automatically.

In [2]:
wavelengths = np.array([600, 620, 640, 660, 680])

emission = fluopy.Spectrum.from_arrays(
    wavelengths=wavelengths,
    values=[0.0, 0.2, 1.0, 0.6, 0.1],
)
absorption_s0 = fluopy.Spectrum.from_arrays(
    wavelengths=wavelengths,
    values=[10000, 40000, 80000, 30000, 5000],
)

custom_data = fluopy.FluorophoreData(
    QUANTUM_YIELD=0.6,
    FLUORESCENCE_LIFETIME=3e-9,
    emission_spectrum=emission,
    absorption_spectra={"s0": absorption_s0},
)

custom_fluorophore = fluopy.Fluorophore(
    name="custom",
    position=[0, 0],
    constants=custom_data,
)

fluorophore_system = fluopy.FluorophoreSystem(
    fluorophores=[custom_fluorophore],
)
transitions = fluorophore_system.load_transitions(
    wavelength=640,
    energy_transfer=False,
    dstorm=False,
)

In [4]:
spectrum_dir = Path(fluopy.__file__).parent / "fluorophore_spectra" / "atto643_data"

custom_data = fluopy.FluorophoreData(
    QUANTUM_YIELD=0.6,
    FLUORESCENCE_LIFETIME=3e-9,
    emission_spectrum=fluopy.Spectrum.from_csv(spectrum_dir / "emission.csv"),
    absorption_spectra={
        "s0": fluopy.Spectrum.from_csv(spectrum_dir / "absorption_s0.csv"),
    },
)

## Adding a single state transition
1. Check if the involved photophysical states are present in transitions.py SingleState, if not, add them
2. Add the transition as a transition.py - TransitionType
3. For automatic read-in, add the rate constant to fluo_data.py (to base class with value 0 and to class instance with true value). Add the transition to transitions.py - derive_transitions(). If the rate depends on other factors, add appropriate constants to fluo_data.py**, look for an appropriate function in formulas.py and if not available, add it. Call the function in derive_transitions() to get the rate constant.

## Adding an ET transition
1. Check if the involved photophysical states are present in transitions.py SingleState and if the PairedState exists, if not, add them
2. Add the transition as a transition.py - TransitionType
3. For automatic read-in, provide absorption spectra* of the acceptor state and emission spectra* for the donor state. Add the transition to transitions.py - derive_energy_transfer_transitions(). 

## Keep in mind

### Spectra
A spectrum consists of one-dimensional wavelength and value arrays. Wavelengths
are given in nm and must be strictly increasing. Spectrum values must be non-negative.

Absorption spectra contain absolute molar extinction coefficients. Emission
spectra may contain relative intensities because they are normalized where
required.

CSV files use 'Wavelengths' and 'y' as the default column names. Alternative
column names can be passed to Spectrum.from_csv().

## Constants
If a cross section is provided, it should correspond to the excitation wavelength used. Energy transfers refer to absorption spectra, not individual cross sections.